# 4.2: Gradients

Read the update for the center word's vector: gradient descent moves $`\mathbf{v}_c`$ toward the real context word's output vector, with a strength $`1 - s_o`$ that shrinks as the model becomes confident, and away from each negative's output vector, with strength $`s_i`$, which is large only for negatives the model currently mistakes for real. Well-classified pairs contribute almost nothing, which is exactly the behavior we want.

As in Chapter 2, let us check the derivation against autograd:


In [ ]:
import torch
import torch.nn.functional as F
torch.manual_seed(0)
torch.set_default_dtype(torch.float64)

d, k = 8, 5
v_c = torch.randn(d, requires_grad=True)        # center word, input vector
u_o = torch.randn(d, requires_grad=True)        # true context word, output vector
u_neg = torch.randn(k, d, requires_grad=True)   # k negative samples, output vectors

loss = -F.logsigmoid(u_o @ v_c) - F.logsigmoid(-(u_neg @ v_c)).sum()
loss.backward()

with torch.no_grad():                           # the formulas derived above
    s_pos = torch.sigmoid(u_o @ v_c)
    s_neg = torch.sigmoid(u_neg @ v_c)
    g_vc = (s_pos - 1) * u_o + (s_neg[:, None] * u_neg).sum(0)
    g_uo = (s_pos - 1) * v_c
    g_neg = s_neg[:, None] * v_c

print(f"loss = {loss.item():.4f}")
for name, ours, auto in [("v_c", g_vc, v_c.grad), ("u_o", g_uo, u_o.grad), ("u_neg", g_neg, u_neg.grad)]:
    print(f"{name:5s}: max |difference| = {(ours - auto).abs().max().item():.1e}")


**Expected output**

Output:

```text
loss = 7.2111
v_c  : max |difference| = 2.2e-16
u_o  : max |difference| = 0.0e+00
u_neg: max |difference| = 4.4e-16
```
